# 03_grid_events_autoloader

**Purpose:** This notebook is reserved for loading grid event CSV files into the Bronze layer.
**Source domain**: `sample_data/grid_events/`
**Expected Bronze table:** `vattenfall_dev.raw.bronze_grid_events`

**Expected fields**
- event_id
- event_date
- region
- asset_id
- event_type
- severity
- duration_minutes
- source_system

In [0]:

from pyspark.sql import functions as F

catalog = "vattenfall_dev"
schema = "raw"

source_domain = "grid_events"

landing_path = f"/Volumes/{catalog}/{schema}/landing/grid_events"
checkpoint_path = f"/Volumes/{catalog}/{schema}/checkpoints/grid_events_checkpoint"
schema_path = f"/Volumes/{catalog}/{schema}/checkpoints/grid_events_schema"
bronze_table = f"{catalog}.{schema}.bronze_grid_events"

print("Source domain:", source_domain)
print("Landing path:", landing_path)
print("Checkpoint path:", checkpoint_path)
print("Schema path:", schema_path)
print("Bronze target table:", bronze_table)

display(dbutils.fs.ls(landing_path))

grid_events_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(landing_path)
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

query = (
    grid_events_stream_df
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

bronze_df = spark.table(bronze_table)

print("Rows in bronze after ingestion:", bronze_df.count())
display(bronze_df.limit(20))